<a href="https://colab.research.google.com/github/MetodiPetrovski/Robotics-and-ML-Tutor-LLM/blob/main/ML_research_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests

url = "https://raw.githubusercontent.com/MetodiPetrovski/Fine-Tuning-Project/main/dataset.json"

data = requests.get(url).json()

In [ ]:
data[-20:]

In [ ]:
!pip install unsloth transformers accelerate peft trl bitsandbytes gradio matplotlib

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-3B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = False,
)

In [ ]:
from datasets import Dataset

def format_example(example):
    return {
        "text": f"<s>[INST] {example['instruction']} [/INST]\n{example['output']}</s>"
    }

dataset = Dataset.from_list(data).map(format_example)

In [ ]:
from transformers import TrainerCallback

vram_usage = []
peak_vram = []
class VRAMCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        vram = torch.cuda.memory_allocated() / 1e9
        vram_usage.append(vram)
        peak_vram.append(torch.cuda.max_memory_allocated() / 1e9)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = "outputs",
    ),
)
trainer.add_callback(VRAMCallback())
trainer.train()


In [ ]:
import matplotlib.pyplot as plt


plt.plot(vram_usage)
plt.xlabel("Step")
plt.ylabel("VRAM (GB)")
plt.title("VRAM Usage During Training")
plt.show()

In [ ]:
def chat_with_model(message, history):
    history = history[-2:]

    conversation = ""

    for user, bot in history:
        conversation += f"[INST] {user} [/INST] {bot} "

    conversation += f"[INST] {message} [/INST]"

    inputs = tokenizer(conversation, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response.split("[/INST]")[-1].strip()

In [ ]:
 import gradio as gr

 demo = gr.ChatInterface(chat_with_model)

 demo.launch(share=True)